# System Threat Classification

Machine learning workflow for predicting whether a system is at risk of a malware threat. The notebook examines missing values, feature types, distributions, and relationships before preparing features and comparing classification models.

## Data

This project uses the **System-Threat-Forecaster** train and test CSV files. Download them from the Kaggle competition.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Loading the Train and Test data

In [2]:
train= pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
data= train.copy()
test_data=test.copy()

###  Shape and Data Samples

In [ ]:
print(data.shape)
print(test.shape)
data.head(10)

**MachineID can be removed, not significant**

In [ ]:
data=data.drop(columns=["MachineID"],axis=1)

# Exploratory Data Analysis

# **Data Statistics**

In [ ]:
data.info()

### It seems many features has missing values in it

In [ ]:
data.describe()

In [ ]:
data.describe(include=["object"])

# Exploring null values

In [ ]:
print("NaN missing values:",data.isnull().sum().sum())
print("Empty missing values:",(data == ' ').sum().sum())
print("Other missing values:",(data.isin(['NA', 'N/A', '-',"?"])).sum().sum())
print("UNKNOWN missing values:",(data == 'UNKNOWN').sum().sum())
print("Unknown missing values:",(data == 'Unknown').sum().sum())

### Which all features has null values and how much??

In [ ]:
null_information=data.isnull().sum()
null_data=null_information.iloc[lambda x:x.values != 0]
null_df= pd.DataFrame(null_data)
null_df.columns=["Null_values"]
null_df=null_df.sort_values(by="Null_values")
print(null_df.shape)
print(null_df)

#### 33 features has null values in it

# Exploring Unique Values

In [ ]:
unique_data=data.nunique()
unique_df=pd.DataFrame(unique_data)
unique_df.columns=['Unique Values']
unique_df['Unique Values'].plot(kind='line', figsize=(8, 4), title='Unique Values')
unique_df=unique_df.sort_values(by="Unique Values")
print(unique_df)

### Some features has only one unique value that is zero which can be removed

# Let's analyze categorical and numerical features invidually

## Exploring Categorical features

In [ ]:
cat_features= data.select_dtypes(include=["object"]).copy()
print("Number of Categorical Features:",cat_features.shape[1])
cat_features.info()

### Only 3 of the categorical features has missing values in it

# What I want from each Categorical Variable? 
*    Unique values,
*    Number of unique values
*    Count including NaN
*    Count excluding NaN
*    Number of missing values
*    Percentage to missing values.
Create a Dataframe to analyze

In [ ]:
cat_stats = pd.DataFrame([
    {
        'Name': c,
        'values': cat_features[c].unique(),
        'values_count_in': cat_features[c].nunique(dropna=False),
        'values_count_ex': cat_features[c].nunique(),
        'miss_values': cat_features[c].isna().sum(),
        '%_miss': round(cat_features[c].isna().mean() * 100, 3),
        "UNKNOWN_values": cat_features[c][cat_features[c] == "UNKNOWN"].count(),
        "Unknown_values": cat_features[c][cat_features[c] == "Unknown"].count()

    }
    for c in cat_features.columns
])
cat_stats = cat_stats.sort_values(by='values_count_in')
cat_stats

### Which all categorical features has missing values in it?

In [ ]:
cat_stats.sort_values(by = '%_miss', ascending = False).head()

### Which all categorical features has unknown values in it?

In [ ]:
cat_stats.sort_values(by=["UNKNOWN_values", "Unknown_values"], ascending=False).head(7)

# Visualization for categorical Features

In [ ]:
for col in cat_features:
    print("col")
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    top_categories = data[col].value_counts().index[:15]

    sns.countplot(data=data, x=col, order=top_categories, palette='Set2', ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis='x', rotation=45)

    sns.countplot(data=data, x=col, hue='target', order=top_categories, palette='Set2', ax=axes[1])
    axes[1].set_title(f"Distribution of {col} by Target")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

### The distribution of categories in many features is biased, but the proportion of target is not that biased
### Do test have the similar distribution????

In [ ]:
for col in cat_features:
    print("col")
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    top_categories = data[col].value_counts().index[:15]

    sns.countplot(data=data, x=col, order=top_categories, palette='Set2', ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis='x', rotation=45)

    sns.countplot(data=test_data, x=col , order=top_categories, palette='Set2', ax=axes[1])
    axes[1].set_title(f"Distribution of {col} by Target")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

### it seems the test data and data has similar distribution only. YAYYY!!!

### Proportional Distribution

In [ ]:
import pandas as pd
for col in cat_features:
    plt.figure(figsize=(8, 4))
    cat_counts = data.groupby([col, 'target']).size().unstack()
    cat_counts.div(cat_counts.sum(axis=1), axis=0).plot(kind="bar", stacked=True, colormap="Set2")
    plt.title(f"Proportional Distribution of {col} by Target")
    plt.xlabel(col)
    plt.ylabel("Proportion")
    plt.legend(title="Target")
    plt.xticks(rotation=45)
    plt.show()

###  Chi-Square Test for Categorical Variables to test their significance

In [ ]:
from scipy.stats import chi2_contingency
alpha=0.05
significant_features = []
for col in cat_features:
    contingency_table = pd.crosstab(data[col], data['target'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    if p < alpha:
        significant_features.append({'column': col, 'chi2': round(chi2,4), 'p_value': round(p,4), 'significant': True})
    else:
        significant_features.append({'column': col, 'chi2': round(chi2,4),'p_value': round(p,4), 'significant': False})
Significant_cat_features = pd.DataFrame(significant_features)
Significant_cat_features

### Which features are significant and which features are not??

In [ ]:
Insignificant_Features = Significant_cat_features[Significant_cat_features['significant']==False]["column"].tolist()
cat_significant_Features = Significant_cat_features[Significant_cat_features['significant']==True]["column"].tolist()
Insignificant_Features

# Insights gained from analyzing the features

# Analyzing Numerical Data

In [ ]:
num_features= data.select_dtypes(exclude=["object"])

num_features=num_features.drop(columns=["target"],axis=1)

num_features

### Looks there are some zero variance features

In [ ]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold()
selected_features = selector.fit_transform(num_features)
feature_mask = selector.get_support()
keep_features = num_features.columns[feature_mask]
remove_features = num_features.columns[~feature_mask]
print("Removed Features:", remove_features.tolist())
num_features=num_features.drop(columns=remove_features,axis=1)

# What I want from each Numerical Variable? 
*    Unique values,
*    Number of unique values
*    Count including NaN
*    Count excluding NaN
*    Number of missing values
*    Percentage to missing values.
Create a Dataframe to for all of this

In [ ]:
num_stats = pd.DataFrame([
    {
        'Name': n,
        'values': num_features[n].unique(),
        'values_count_in': num_features[n].nunique(dropna=False),
        'values_count_ex': num_features[n].nunique(),
        'miss_values': num_features[n].isna().sum(),
        '%_miss': round(num_features[n].isna().mean() * 100, 3),


    }
    for n in num_features.columns
])
num_stats = num_stats.sort_values(by='values_count_in',ignore_index=True)
num_stats

### Which all feature has null values and how much??

In [ ]:
num_stats.sort_values(by = '%_miss', ascending = False).head(35)

# OMG!!! Though some are numerical features they are generally ID's. Lets break them down

In [ ]:
ID_num_stats = num_stats[num_stats['Name'].str.contains("ID")]
for c in ID_num_stats["Name"]:
    data[c]= data[c].astype(str)
id_num_list= ID_num_stats["Name"].tolist()
ID_num_stats

In [ ]:
nonID_num_stats = num_stats[~num_stats['Name'].str.contains("ID")]
nonID_num_stats

## !!! still there are some categorical features in the non_num_id

## Final three segment of categorical features( binary, non_binary, categorical but given as numerical .
### they are: ["OSProductSuite","RegionIdentifier","RealTimeProtectionState","OSBuildNumber","OSBuildNumberOnly"]

In [ ]:
id_num_feature = ID_num_stats["Name"].tolist()
cat_num_feature=id_num_feature+["OSProductSuite","RegionIdentifier","RealTimeProtectionState","OSBuildNumber","OSBuildNumberOnly"]

### Some columns are binary and some are not

given less than five because some features has np.nan and EnableLUA has an error category in it

In [ ]:
binary_num_features= nonID_num_stats[nonID_num_stats["values_count_in"]<5]["Name"].tolist()
nonbinary_num_features =nonID_num_stats[nonID_num_stats["values_count_in"]>5]["Name"].tolist()

## Categorized Numerical Features list names:
* cat_num_feature
* binary_num_features
* nonbinary_num_features

INSIGHTS: OSbuild number and OSbuild number only are same, remove either one

features like regionalidentifier, RealTimeProtectionState ,OSProductSuite , osbuildnumber should also be categorical

convert resolution veritical and horizontal to a single feature and scale
convert mb to gb

#### Inference: These numerical values are very similar to categorical variable.

## HeatMAP

In [ ]:
num=binary_num_features+nonbinary_num_features

In [ ]:
plt.figure(figsize=(20, 15))
sns.heatmap(data[num].corr(),annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

### Highly coorelated features: 
#### * Buildnumber and buildnumber
#### * resolution horizontal and resolution vertical 
#### * Realtimetimeprotection state and ispassive enabled

### Histogram for train and test

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_histograms(train_df, test_df):
    numerical_cols = train_df.columns.tolist()

    for col in numerical_cols:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(train_df[col], kde=True, ax=axes[0], color="blue")
        axes[0].set_title(f"Train Histogram: {col}")
        sns.histplot(test_df[col], kde=True, ax=axes[1], color="red")
        axes[1].set_title(f"Test Histogram: {col}")
        plt.tight_layout()
        plt.show()


plot_histograms(train_df=data[num], test_df=test_data[num])

In [ ]:
def plot_violinplots(train_df, test_df):
    numerical_cols = train_df.columns.tolist()

    for col in numerical_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.violinplot(x=train_df[col], ax=axes[0], color="blue")
        axes[0].set_title(f"Train Violin Plot: {col}")
        sns.violinplot(x=test_df[col], ax=axes[1], color="red")
        axes[1].set_title(f"Test Violin Plot: {col}")
        plt.tight_layout()
        plt.show()

plot_violinplots(train_df=data[num], test_df=test_data[num])

In [ ]:
def plot_boxplots(train_df, test_df):
    numerical_cols = train_df.columns.tolist()

    for col in numerical_cols:

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.boxplot(x=train_df[col], ax=axes[0], color="blue")
        axes[0].set_title(f"Train Boxplot: {col}")
        sns.boxplot(x=test_df[col], ax=axes[1], color="red")
        axes[1].set_title(f"Test Boxplot: {col}")
        plt.tight_layout()
        plt.show()

plot_boxplots(train_df=data[nonbinary_num_features], test_df=test_data[nonbinary_num_features])

# The test data also has outliers
not a better plan to remove the outliers

### Significant categorical features but given as numerical

In [ ]:
from scipy.stats import chi2_contingency
alpha=0.05
significant_numfeatures = []
for col in cat_num_feature:
    contingency_table = pd.crosstab(data[col], data['target'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    if p < alpha:
        significant_numfeatures.append({'column': col, 'chi2': round(chi2,4), 'p_value': round(p,4), 'significant': True})
    else:
        significant_numfeatures.append({'column': col, 'chi2': round(chi2,4),'p_value': round(p,4), 'significant': False})
Significant_num_features = pd.DataFrame(significant_numfeatures)

In [ ]:
Significant_num_features

### Lets transform them as objects

In [ ]:
for c in cat_num_feature:
    data[c]=data[c].astype('object')

## Lets refine the significant features based on out EDA

In [ ]:
final_cat_features=data.select_dtypes(include=["object"])
opt_cat_features =final_cat_features.columns.tolist().copy()
x=['ProductName', 'OSVersion', 'FlightRing',"MachineID","OSBuildNumberOnly"]
for i in opt_cat_features:
    if i in x:
        opt_cat_features.remove(i)
opt_cat_features

In [ ]:
print(data.columns.tolist())

In [ ]:
import numpy as np

data["PPI"] = np.sqrt((data["PrimaryDisplayResolutionHorizontal"]**2 + data["PrimaryDisplayResolutionVertical"]**2) / data["PrimaryDisplayDiagonalInches"])


data["TotalPhysicalRAMMB"] /= 1024

data["PrimaryDiskCapacityMB"] /= 1024

data["SystemVolumeCapacityMB"] /= 1024

data.rename(columns={
    "TotalPhysicalRAMMB": "TotalPhysicalRAMGB",
    "PrimaryDiskCapacityMB": "PrimaryDiskCapacityGB",
    "SystemVolumeCapacityMB": "SystemVolumeCapacityGB"
}, inplace=True)


data.loc[data["InternalBatteryNumberOfCharges"] > 100000, "InternalBatteryNumberOfCharges"] = np.nan

In [ ]:
final_num_features=data.select_dtypes(exclude=["object"])
opt_num_features =final_num_features.columns.tolist().copy()
x=['PrimaryDisplayDiagonalInches','PrimaryDisplayResolutionHorizontal','PrimaryDisplayResolutionVertical','IsBetaUser', 'AutoSampleSubmissionEnabled', 'IsFlightsDisabled','target']
for i in opt_num_features:
    if i in x:
        opt_num_features.remove(i)
opt_num_features

opt_bin_num_features=[]
opt_nonbin_num_features=[]
extra=[]
for i in opt_num_features:
    if i in binary_num_features:
        opt_bin_num_features.append(i)
    else:

        opt_nonbin_num_features.append(i)


print(len(opt_num_features))
print(len(opt_bin_num_features))
print(len(opt_nonbin_num_features))

## categorizing categorical features

In [ ]:
import pandas as pd

cat_count = []

for c in opt_cat_features:
    cat_count.append({"col": c, "count": final_cat_features[c].nunique(dropna=False)})

cat_count_df = pd.DataFrame(cat_count)

one_hot=(cat_count_df[cat_count_df['count']<=20])['col'].values.tolist()
print(one_hot)

target_enc= (cat_count_df[cat_count_df['count']>20])['col'].values.tolist()
print(target_enc)

# *Data Preprocessing*

In [ ]:
X=data.drop(columns=["target"])
y=data["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    stratify = y,
                                                    random_state = 42)

In [ ]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder,MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector as mcs
from category_encoders import TargetEncoder

## Pipelines

## Categorical Piplines

In [ ]:
cat_simple_onehot =  Pipeline([("imputer", SimpleImputer(missing_values = np.nan, strategy="most_frequent")),
                       ("one_hot_encoder", OneHotEncoder(handle_unknown='ignore',sparse_output=False))])

cat_simple_target =  Pipeline([("imputer", SimpleImputer(missing_values = np.nan, strategy="most_frequent")),
                       ("Target_encoder", TargetEncoder())])

## Numerical Pipelines

In [ ]:
num_simple_mean_robust = Pipeline([("imputer", SimpleImputer(missing_values = np.nan, strategy="most_frequent")),
                       ("r_scaler", RobustScaler())])

num_median_minmax = Pipeline([("imputer", SimpleImputer(missing_values = np.nan, strategy="median")),
                       ("r_scaler", MinMaxScaler())])

num_freq_simple = Pipeline([("imputer",SimpleImputer(strategy="most_frequent"))])


num_median_standard = Pipeline([("imputer", SimpleImputer(missing_values = np.nan, strategy="median")),
                       ("r_scaler", StandardScaler())])

id_num_onehot =       Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                                ("encoder",OneHotEncoder(sparse_output=False,handle_unknown='ignore'))])

id_num_target =       Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                                ("encoder",TargetEncoder())])

In [ ]:
ct = ColumnTransformer([("Small_Cat",cat_simple_onehot,one_hot),
                        ("Large_Cat",cat_simple_target,target_enc),
                        ("binary_imp",num_freq_simple,opt_bin_num_features),
                        ("nonbinary_imp",num_median_standard, opt_nonbin_num_features)],
                        remainder='drop', verbose_feature_names_out=False)


ct.fit(X_train, y_train)

X_train_trf = ct.transform(X_train)
X_test_trf = ct.transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_curve, roc_auc_score, auc,confusion_matrix,make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV

# Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(random_state=42)

lr.fit(X_train_trf, y_train)

lr_y_hat_train = lr.predict(X_train_trf)
lr_y_hat_val = lr.predict(X_test_trf)

lr_train_score = accuracy_score(y_train, lr_y_hat_train)
lr_val_score = accuracy_score(y_test, lr_y_hat_val)

print(f"Training Score: {lr_train_score}, Validation Score: {lr_val_score}")
print("\nClassification Report:\n", classification_report(y_test, lr_y_hat_val))

# Hypertuning for Logistic Regression

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform

param_grid_lr = {
    "C": uniform(0.01, 10),
    "solver": ["liblinear", "lbfgs"],
    "max_iter": [500, 1000]
}


lr = LogisticRegression(random_state=42)

random_search_lr = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_grid_lr,
    n_iter=10,
    scoring="accuracy",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


random_search_lr.fit(X_train_trf, y_train)

best_lr = random_search_lr.best_estimator_
print(f"Best Params for Logistic Regression: {random_search_lr.best_params_}")

y_pred_lr = best_lr.predict(X_test_trf)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)}")

param_grid_lr = {
    "C": [0.5, 0.59, 0.6, 0.7],
    "solver": ["liblinear"],
    "max_iter": [1000]
}

lr = LogisticRegression(random_state=42)

grid_search_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    scoring="accuracy",
    cv=5,  # More stable results
    verbose=1,
    n_jobs=-1
)

grid_search_lr.fit(X_train_trf, y_train)

best_lr = grid_search_lr.best_estimator_
print(f"Best Params for Logistic Regression: {grid_search_lr.best_params_}")

y_pred_lr = best_lr.predict(X_test_trf)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)}")

# DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_model=DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_trf,y_train)
dt_y_hat_train=dt_model.predict(X_train_trf)
dt_y_hat_val=dt_model.predict(X_test_trf)
dt_train_score = accuracy_score(dt_y_hat_train, y_train)
dt_val_score = accuracy_score(dt_y_hat_val, y_test)
print(f'Training score: {dt_train_score}, Validation score: {dt_val_score}')

# HyperParameter Tuning for DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist_dt = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": np.arange(3, 50, 5),
    "min_samples_split": np.arange(2, 20, 2),
    "min_samples_leaf": np.arange(1, 20, 2),
}

dt = DecisionTreeClassifier(random_state=42)

random_search_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist_dt,
    n_iter=30,
    scoring="accuracy",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search_dt.fit(X_train_trf, y_train)

best_dt = random_search_dt.best_estimator_
print(f"Best Params for Decision Tree (Randomized Search): {random_search_dt.best_params_}")


y_pred_dt = best_dt.predict(X_test_trf)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt)}")
from sklearn.model_selection import GridSearchCV


param_grid_dt = {
    "criterion": [random_search_dt.best_params_["criterion"]],
    "max_depth": [random_search_dt.best_params_["max_depth"] - 5,
                  random_search_dt.best_params_["max_depth"],
                  random_search_dt.best_params_["max_depth"] + 5],
    "min_samples_split": [max(2, random_search_dt.best_params_["min_samples_split"] - 1),
                          random_search_dt.best_params_["min_samples_split"],
                          random_search_dt.best_params_["min_samples_split"] + 1],  # Ensuring it's ≥ 2
    "min_samples_leaf": [max(1, random_search_dt.best_params_["min_samples_leaf"] - 1),
                         random_search_dt.best_params_["min_samples_leaf"],
                         random_search_dt.best_params_["min_samples_leaf"] + 1]

}


grid_search_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    scoring="accuracy",
    cv=5,
    verbose=1,
    n_jobs=-1
)


grid_search_dt.fit(X_train_trf, y_train)


best_dt_grid = grid_search_dt.best_estimator_
print(f"Best Params for Decision Tree (Grid Search): {grid_search_dt.best_params_}")

y_pred_dt_grid = best_dt_grid.predict(X_test_trf)
print(f"Decision Tree Accuracy (Grid Search): {accuracy_score(y_test, y_pred_dt_grid)}")

# RandomForest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_trf, y_train)

rf_y_hat_train = rf.predict(X_train_trf)
rf_y_hat_val = rf.predict(X_test_trf)

rf_train_score = accuracy_score(y_train, rf_y_hat_train)
rf_val_score = accuracy_score(y_test, rf_y_hat_val)
print(f"Training Score: {rf_train_score}, Validation Score: {rf_val_score}")

# HyperTuning for RandomForestClassifier

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

param_dist_rf = {
    'n_estimators': np.arange(100, 500, 100),
    'max_depth': np.arange(10, 50, 10),
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

random_search_rf = RandomizedSearchCV(
    rf, param_distributions=param_dist_rf,
    n_iter=10,
    cv=3,
    verbose=2,
    n_jobs=-1
)

random_search_rf.fit(X_train_trf, y_train)
best_rf = random_search_rf.best_estimator_
print(f"Best Params for Random Forest: {random_search_rf.best_params_}")

y_pred_rf = best_rf.predict(X_test_trf)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)}")

In [ ]:
best_rf = random_search_rf.best_estimator_
print(f"Best Params for Random Forest: {random_search_rf.best_params_}")

y_pred_rf = best_rf.predict(X_test_trf)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)}")

## Best model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf1 = RandomForestClassifier(
    n_estimators=400,
    min_samples_split=10,
    min_samples_leaf=1,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf1.fit(X_train_trf, y_train)

rf1_y_hat_train = rf1.predict(X_train_trf)
rf1_y_hat_val = rf1.predict(X_test_trf)

rf1_train_score = accuracy_score(y_train, rf1_y_hat_train)
rf1_val_score = accuracy_score(y_test, rf1_y_hat_val)
print(f"Training Score: {rf1_train_score}, Validation Score: {rf1_val_score}")

# XGBoost

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1)
xgb.fit(X_train_trf, y_train)
xgb_pred = xgb.predict(X_test_trf)
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

# HyperParameter tuning for XGBoost

In [ ]:
from xgboost import XGBClassifier


param_dist_xgb = {
    'n_estimators': np.arange(100, 500, 100),
    'max_depth': np.arange(3, 10, 2),
    'learning_rate': np.linspace(0.01, 0.3, 10),
    'subsample': np.linspace(0.5, 1.0, 5),
    'colsample_bytree': np.linspace(0.5, 1.0, 5),
    'gamma': np.linspace(0, 0.5, 5),
    'reg_lambda': np.logspace(-1, 2, 5) }
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

random_search_xgb = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring='accuracy',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search_xgb.fit(X_train_trf, y_train)

print("Best parameters:", random_search_xgb.best_params_)
print("Best score:", random_search_xgb.best_score_)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

xgb_best = XGBClassifier(
    subsample=0.5,
    reg_lambda=0.1,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.07444444444444444,
    gamma=0.125,
    colsample_bytree=0.875,
    random_state=42,
    n_jobs=-1
)

xgb_best.fit(X_train_trf, y_train)

xgb_train_preds = xgb_best.predict(X_train_trf)
xgb_val_preds = xgb_best.predict(X_test_trf)


xgb_train_acc = accuracy_score(y_train, xgb_train_preds)
xgb_val_acc = accuracy_score(y_test, xgb_val_preds)
print(f"Training Accuracy: {xgb_train_acc:.4f}, Validation Accuracy: {xgb_val_acc:.4f}")

# LGBMClassifier

In [ ]:
lgbm = LGBMClassifier(random_state=42, n_jobs=-1)
lgbm.fit(X_train_trf, y_train)
lgbm_pred = lgbm.predict(X_test_trf)
print("LightGBM Accuracy:", accuracy_score(y_test, lgbm_pred))

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'n_estimators': np.arange(100, 500, 100),
    'learning_rate': np.linspace(0.01, 0.3, 10),
    'max_depth': np.arange(3, 10, 2),
    'num_leaves': np.arange(20, 150, 20),
    'min_child_samples': np.arange(10, 100, 10),
    'subsample': np.linspace(0.5, 1, 5),
    'colsample_bytree': np.linspace(0.5, 1, 5),
    'reg_alpha': np.logspace(-3, 1, 5),
    'reg_lambda': np.logspace(-3, 1, 5),
}

# Initialize model
lgbm = LGBMClassifier(random_state=42)

# Perform RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train_trf, y_train)

print("Best parameters found: ", random_search.best_params_)
print("Best accuracy: ", random_search.best_score_)

In [ ]:
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(random_state=42, n_jobs=-1))
]
meta_model = LogisticRegression(max_iter=1000, random_state=42)
stacking_clf = StackingClassifier(estimators=base_estimators, final_estimator=meta_model, n_jobs=-1)
stacking_clf.fit(X_train_trf, y_train)
stack_pred = stacking_clf.predict(X_test_trf)
print("Stacking Classifier Accuracy:", accuracy_score(y_test, stack_pred))

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


base_models = [
    ('xgb', XGBClassifier(subsample=0.5, reg_lambda=0.1, n_estimators=200, max_depth=5,
                          learning_rate=0.0744, gamma=0.125, colsample_bytree=0.875, random_state=42)),

    ('lgbm', LGBMClassifier(subsample=0.625, reg_lambda=1.0, reg_alpha=0.001, num_leaves=60,
                            n_estimators=400, min_child_samples=10, max_depth=3,
                            learning_rate=0.1389, colsample_bytree=0.625, random_state=42)),

    ('rf', RandomForestClassifier(n_estimators=400, min_samples_split=10,
                                  min_samples_leaf=1, max_depth=20, random_state=42, n_jobs=-1))
]

In [ ]:
stack_clf = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(), n_jobs=-1)

param_grid = {
    'final_estimator__C': [0.01, 0.1, 1, 10],
    'final_estimator__solver': ['lbfgs', 'liblinear']
}

grid_search = GridSearchCV(estimator=stack_clf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train_trf, y_train)

print("Best parameters for meta-learner: ", grid_search.best_params_)
print("Best accuracy: ", grid_search.best_score_)

In [ ]:
test["PPI"] = np.sqrt((test["PrimaryDisplayResolutionHorizontal"]**2 + test["PrimaryDisplayResolutionVertical"]**2) / test["PrimaryDisplayDiagonalInches"])


test["TotalPhysicalRAMMB"] /= 1024

test["PrimaryDiskCapacityMB"] /= 1024

test["SystemVolumeCapacityMB"] /= 1024

test.rename(columns={
    "TotalPhysicalRAMMB": "TotalPhysicalRAMGB",
    "PrimaryDiskCapacityMB": "PrimaryDiskCapacityGB",
    "SystemVolumeCapacityMB": "SystemVolumeCapacityGB"
}, inplace=True)

In [ ]:
test_trf =ct.transform(test)
y_pred=rf1.predict(test_trf)
print("success")

In [ ]:
submission=pd.DataFrame({"id":range(0,test.shape[0]),
                        "target":y_pred})
submission.to_csv("submission.csv",index=False)
print("success")

In [ ]:
submission.head(100)